# Predicción de Diabetes (Pima Indians)

Proyecto de clasificación listo para **Google Colab**.

**Pregunta:** ¿podemos predecir si una persona tiene diabetes a partir de indicadores médicos?

**Target:** `Outcome` (1 = diabetes, 0 = no)

**Stack:** pandas, seaborn, scikit-learn, shap

Dataset clásico Pima Indians Diabetes.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Cargar datos

In [ ]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
cols = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]
df = pd.read_csv(url, names=cols)
print(df.shape)
print(df["Outcome"].value_counts())
df.head()

## 2. EDA

In [ ]:
print("Tasa de diabetes:", df["Outcome"].mean().round(3))
df.describe().T

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Outcome")
plt.xticks([0, 1], ["No", "Sí"])
plt.title("Distribución de diabetes")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(data=df, x="Glucose", hue="Outcome", kde=True, ax=axes[0, 0])
sns.histplot(data=df, x="BMI", hue="Outcome", kde=True, ax=axes[0, 1])
sns.histplot(data=df, x="Age", hue="Outcome", kde=True, ax=axes[1, 0])
sns.boxplot(data=df, x="Outcome", y="Glucose", ax=axes[1, 1])
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlación")
plt.show()

### Hallazgos
- ~35% de los casos tienen diabetes.
- `Glucose` es la variable más correlacionada con Outcome.
- Hay valores 0 en varias columnas que representan datos faltantes.

## 3. Preparar datos

In [ ]:
cols_zero = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df[cols_zero] = df[cols_zero].replace(0, np.nan)

X = df.drop(columns="Outcome")
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)
print("Nulos en train:", X_train.isnull().sum().sum())

## 4. Modelos

In [ ]:
from sklearn.impute import SimpleImputer

modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

resultados = []
pipes = {}

for nombre, clf in modelos.items():
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", clf)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    resultados.append({
        "modelo": nombre,
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
    })
    pipes[nombre] = pipe

pd.DataFrame(resultados).sort_values("roc_auc", ascending=False)

## 5. Evaluación

In [ ]:
mejor = "Random Forest"
pipe = pipes[mejor]
y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["No", "Sí"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["No", "Sí"]).plot(ax=axes[0], cmap="Blues")
axes[0].set_title("Matriz de confusión")

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].set_title("Curva ROC")
plt.tight_layout()
plt.show()

## 6. Feature importance

In [ ]:
rf = pipes["Random Forest"].named_steps["model"]
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(7, 4))
sns.barplot(x=importancias.values, y=importancias.index)
plt.title("Importancia — Random Forest")
plt.tight_layout()
plt.show()

## 7. SHAP

In [ ]:
!pip install -q shap

In [ ]:
import shap

rf_pipe = pipes["Random Forest"]
X_train_trans = rf_pipe.named_steps["imputer"].transform(X_train)
X_train_trans = rf_pipe.named_steps["scaler"].transform(X_train_trans)
model = rf_pipe.named_steps["model"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train_trans)

print("SHAP listo")

In [ ]:
shap.summary_plot(shap_values[1], X_train_trans, feature_names=X.columns, show=False)
plt.title("SHAP Summary — diabetes")
plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(shap_values[1], X_train_trans, feature_names=X.columns, plot_type="bar", show=False)
plt.title("Importancia media SHAP")
plt.tight_layout()
plt.show()

## 8. Conclusiones

- `Glucose` es la variable más importante (EDA + importance + SHAP).
- ROC-AUC suele estar en **0.80–0.85** en test.
- Hay que tratar los ceros como datos faltantes.
- Buen proyecto para practicar imputación + clasificación desbalanceada.

**Rango esperado:** Accuracy ~0.75–0.80 | ROC-AUC ~0.82–0.86